# 03a - From-scratch DPO on HH-RLHF (helpful/harmless) (Task 4)
**Accelerator:** GPU T4 x2 (policy on cuda:0, frozen reference on cuda:1) · **Internet:** ON · **Inputs:** outputs of notebooks 01 and 02 · **Runtime:** ~2-2.5 h

Run with **Save Version → Save & Run All (Commit)**.

In [ ]:
# ==== EDIT THIS: your GitHub fork of the repository ====
REPO_URL = "https://github.com/Malik9102/small-model-alignment.git"
REPO_DIR = "/kaggle/working/small-model-alignment"

import glob, os, shutil, subprocess, sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=False)

def sh(command):
    """Run a shell command, stream its output, and STOP the notebook if it fails.
    (A plain `!command` keeps going after an error, so a crashed training run would go unnoticed.)"""
    print(f"$ {command}", flush=True)
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {process.returncode}: {command}")

def find_input(pattern):
    """Locate a file/dir inside attached Kaggle inputs (any dataset/notebook slug)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"Nothing matching {pattern!r} under /kaggle/input - did you attach the right notebook output?")
    if len(hits) > 1:
        print(f"[find_input] {len(hits)} matches for {pattern!r}, using {hits[0]}")
    return hits[0]

In [ ]:
# Keep Kaggle's CUDA-matched PyTorch. Do NOT reinstall torch.
sh(f"""python -m pip uninstall -y -q torchao || true""")
sh(f"""python -m pip install -q "transformers>=4.56.2,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib pytest""")
sh(f"""python -m pip install -q -e . --no-deps""")

In [ ]:
sh(f"""python scripts/env_check.py""")

In [ ]:
# Bring the fixed data files from notebook 01's output into the repo
os.makedirs("data/processed", exist_ok=True)
src = os.path.dirname(find_input("data/processed/manifest.json"))
for name in os.listdir(src):
    shutil.copy(os.path.join(src, name), "data/processed/")
shutil.copy(find_input("eval_sets/alpaca_eval_300_seed42.json"), "eval_sets/")
print(sorted(os.listdir("data/processed")))

In [ ]:
SFT_BEST = os.path.dirname(find_input('sft/best/model.safetensors'))
print('SFT best checkpoint (policy init + frozen reference):', SFT_BEST)
sh(f"""cat {SFT_BEST}/training_state.json | head -c 300""")

## Pilot (3 optimizer steps). Step-0 validation must show loss 0.6931 and reward accuracy 0.000.

In [ ]:
sh(f"""python scripts/train_dpo.py --config configs/dpo_hh.yaml --set sft_checkpoint={SFT_BEST} --set max_steps=3 --set limit_val=32 --set eval_every=3 --set log_every=1 --set save_final=false --set output_dir=/kaggle/working/pilot_dpo""")
shutil.rmtree('/kaggle/working/pilot_dpo', ignore_errors=True)

## Full run: 15,000 pairs, β=0.1, lr 1e-6 constant, effective batch 32, best checkpoint by validation reward accuracy

In [ ]:
sh(f"""python scripts/train_dpo.py --config configs/dpo_hh.yaml --set sft_checkpoint={SFT_BEST} --set output_dir=outputs/dpo_hh""")

In [ ]:
sh(f"""cat outputs/dpo_hh/summary.json""")
sh(f"""python scripts/plot_metrics.py --dpo outputs/dpo_hh --output_dir outputs/plots""")

In [ ]:
from IPython.display import Image
Image('outputs/plots/dpo_hh_training.png')

In [ ]:
sh(f"""python scripts/generate.py --model_path outputs/dpo_hh/best --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator qwen2.5-0.5b-dpo-hh --output outputs/generations/sanity_dpo_hh.json --limit 5 --print_samples --overwrite""")

In [ ]:
# Lightweight archive (metrics/logs/plots/generations - not the 2 GB checkpoints)
sh(f"""cd {REPO_DIR} && zip -qr /kaggle/working/03_dpo_hh_run.zip outputs/dpo_hh outputs/plots -x "*/best/*" "*/final/*" "*/trainer/*" && ls -lh /kaggle/working/03_dpo_hh_run.zip""")
sh(f"""df -h /kaggle/working | tail -1""")